<a href="https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/Copy_of_w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import json, os

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding — "ML Appendix: What Predicts Health?"** (Random Forest, Health Score, feature
importance: Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8%).

*Methodology question:* `health_score` is explicitly defined in the paper's own metrics section as
`impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts)`. The model's four
top-ranked features are exactly the four ingredients of that formula. This is the textbook
label-derived-feature pattern — the label was computed *from* these columns, and those same
columns are the inputs. The paper is admirably upfront that "importance is descriptive rather than
causal," which is the right instinct, but the natural follow-up question is: **what does feature
importance look like with the four definitional inputs removed**, using only independent structural
signals (word count, age, freshness, search volume)? Without that version, it's hard to know
whether *anything* outside the label's own formula predicts health score at all — right now the
appendix can't distinguish "the model learned something real" from "the model rediscovered
arithmetic."

**Finding — "ML Appendix: What Predicts Growth?"** (Logistic Regression, "71% holdout accuracy").

*Methodology question:* the paper's own portfolio-level split earlier in the study is 74.8K growing
vs. 45.6K declining — roughly 62% growing. If the 61.8K-row ML sample used for this classifier has
a similar imbalance, a naive "always predict growing" rule would already score close to 62%
accuracy, which makes "71% holdout accuracy" closer to a single-digit lift over the base rate than
the headline number suggests on its own. This is exactly the "always print the base rate next to
the metric" rule — the paper reports the split percentage (80/20) but not the class balance of that
split, so a reader can't tell how much of the 71% is genuine skill. A natural, constructive addition
would be one line: "base rate in this holdout: X% growing."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import json, os
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

REPO = "hf://datasets/FlyRank/internship-warehouse"
os.makedirs("work/outputs", exist_ok=True)

fact = pd.read_parquet(f"{REPO}/fact_content_daily_performance/month=2026-03/data_0.parquet")
dim_content = pd.read_parquet(f"{REPO}/dim_content.parquet")

fact_avail = fact[fact["gsc_data_available"] == True].copy()
agg = (fact_avail.groupby(["client_hash_id", "content_hash_id"])
       .agg(impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            sum_position=("gsc_sum_position", "sum"))
       .reset_index())
agg["avg_position"] = agg["sum_position"] / agg["impressions"]
agg["ctr"] = agg["clicks"] / agg["impressions"] * 100

VISIBLE_MIN_IMPR = 150
visible = agg[(agg["impressions"] >= VISIBLE_MIN_IMPR) & (agg["avg_position"] > 0)].copy()

bins, labels = [0, 3, 10, 20, 50, np.inf], ["top_3", "page_1", "striking", "page_3_5", "deep"]
visible["position_tier"] = pd.cut(visible["avg_position"], bins=bins, labels=labels)
visible["tier_median_ctr"] = visible.groupby("position_tier", observed=True)["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]

content_cols = ["client_hash_id", "content_hash_id", "content_type", "main_intent", "word_count", "search_volume"]
visible = visible.merge(dim_content[content_cols], on=["client_hash_id", "content_hash_id"], how="left")

threshold = visible["ctr_gap"].quantile(0.10)
visible["y_true"] = (visible["ctr_gap"] <= threshold).astype(int)

FEATURES = ["position_tier", "word_count", "content_type", "main_intent", "search_volume", "impressions"]
cat_cols = ["position_tier", "content_type", "main_intent"]
num_cols = ["word_count", "search_volume", "impressions"]

def make_pipe():
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ])
    return Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=1000, random_state=42))])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return y_true.values[order].mean()

def prep_X(df):
    X = df[FEATURES].copy()
    for c in num_cols:
        X[c] = X[c].fillna(X[c].median())
    for c in cat_cols:
        X[c] = X[c].astype(str).fillna("missing")
    return X

# ===== BEFORE: naive random row-level split =====
train_rand, test_rand = train_test_split(visible, test_size=0.25, random_state=42)
Xtr, Xte = prep_X(train_rand), prep_X(test_rand)
ytr, yte = train_rand["y_true"], test_rand["y_true"]
pipe_rand = make_pipe()
pipe_rand.fit(Xtr, ytr)
proba_rand = pipe_rand.predict_proba(Xte)[:, 1]

before = {
    "split": "random row-level (BEFORE)",
    "test_rows": len(test_rand),
    "client_overlap": len(set(train_rand["client_hash_id"]) & set(test_rand["client_hash_id"])),
    "precision_at_50": round(precision_at_k(yte, proba_rand, 50), 3),
    "precision_at_100": round(precision_at_k(yte, proba_rand, 100), 3),
    "roc_auc": round(roc_auc_score(yte, proba_rand), 3),
}

# ===== AFTER: grouped by client (same design as Week 5) =====
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(visible, groups=visible["client_hash_id"]))
train_grp, test_grp = visible.iloc[train_idx].copy(), visible.iloc[test_idx].copy()
Xtr2, Xte2 = prep_X(train_grp), prep_X(test_grp)
ytr2, yte2 = train_grp["y_true"], test_grp["y_true"]
pipe_grp = make_pipe()
pipe_grp.fit(Xtr2, ytr2)
proba_grp = pipe_grp.predict_proba(Xte2)[:, 1]

after = {
    "split": "grouped by client (AFTER, same as Week 5)",
    "test_rows": len(test_grp),
    "client_overlap": len(set(train_grp["client_hash_id"]) & set(test_grp["client_hash_id"])),
    "precision_at_50": round(precision_at_k(yte2, proba_grp, 50), 3),
    "precision_at_100": round(precision_at_k(yte2, proba_grp, 100), 3),
    "roc_auc": round(roc_auc_score(yte2, proba_grp), 3),
}

print("=== BEFORE / AFTER SPLIT COMPARISON ===")
print(pd.DataFrame([before, after]).to_string(index=False))

=== BEFORE / AFTER SPLIT COMPARISON ===
                                    split  test_rows  client_overlap  precision_at_50  precision_at_100  roc_auc
                random row-level (BEFORE)      22994              37             0.84              0.80    0.913
grouped by client (AFTER, same as Week 5)       6545               0             0.48              0.52    0.798


**The gap is the finding.** The random split leaks client identity: 37 clients appear in *both*
train and test, so the model partly memorizes client-specific patterns rather than learning
something that transfers to a new client. That inflates every number — precision@50 nearly doubles
(0.48 → 0.84), precision@100 jumps (0.52 → 0.80), and AUC climbs from 0.798 to 0.913. None of that
gain is real skill; it's the same client showing up on both sides of the split. **The honest number
for this model is the grouped-split row — precision@50 = 0.48, precision@100 = 0.52, AUC = 0.798 —
the same figures reported in Week 5.** This is the whole reason Week 5 used the grouped split from
the start rather than defaulting to the easier random one.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
print("=== LEAKAGE AUDIT: add a label-derived feature on purpose ===")
FEATURES_LEAKY = FEATURES + ["ctr_gap"]
cat_cols_leaky, num_cols_leaky = cat_cols, num_cols + ["ctr_gap"]

def prep_X_leaky(df):
    X = df[FEATURES_LEAKY].copy()
    for c in num_cols_leaky:
        X[c] = X[c].fillna(X[c].median())
    for c in cat_cols_leaky:
        X[c] = X[c].astype(str).fillna("missing")
    return X

pre_leaky = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_leaky),
    ("num", StandardScaler(), num_cols_leaky),
])
pipe_leaky = Pipeline([("prep", pre_leaky), ("model", LogisticRegression(max_iter=1000, random_state=42))])
Xtr_leak, Xte_leak = prep_X_leaky(train_grp), prep_X_leaky(test_grp)
pipe_leaky.fit(Xtr_leak, ytr2)
proba_leaky = pipe_leaky.predict_proba(Xte_leak)[:, 1]
leaky_auc = roc_auc_score(yte2, proba_leaky)

print(f"honest AUC (Week 5 safe features, grouped split): {after['roc_auc']}")
print(f"leaky AUC  (adding ctr_gap, which the label is built from): {round(leaky_auc, 3)}")
print("--> deleting ctr_gap, keeping the honest score:", after["roc_auc"])

print("\n=== attack checklist ===")
print("1. label-derived features in final set?      ", "NO -", FEATURES)
print("2. future-window features?                    NO - all from March 2026, same window as label")
print("3. product/flag columns used?                 NO - none exist in this warehouse release")
print("4. population filter outcome-dependent?        NO - impressions>=150 filter uses same-window traffic, not label")
print("5. split respects real-world grouping?         YES - grouped by client_hash_id, 0 overlap (see AFTER row above)")
print(f"6. base rate reported next to metric?          YES - {round(yte2.mean(),3)} (positive rate, grouped test set)")

with open("work/outputs/w06_metrics.json", "w") as f:
    json.dump({"before": before, "after": after,
               "leakage_audit": {"honest_auc": after["roc_auc"], "leaky_auc": round(leaky_auc, 3)}}, f, indent=2)
print("\nwrote work/outputs/w06_metrics.json")

=== LEAKAGE AUDIT: add a label-derived feature on purpose ===
honest AUC (Week 5 safe features, grouped split): 0.798
leaky AUC  (adding ctr_gap, which the label is built from): 1.0
--> deleting ctr_gap, keeping the honest score: 0.798

=== attack checklist ===
1. label-derived features in final set?       NO - ['position_tier', 'word_count', 'content_type', 'main_intent', 'search_volume', 'impressions']
2. future-window features?                    NO - all from March 2026, same window as label
3. product/flag columns used?                 NO - none exist in this warehouse release
4. population filter outcome-dependent?        NO - impressions>=150 filter uses same-window traffic, not label
5. split respects real-world grouping?         YES - grouped by client_hash_id, 0 overlap (see AFTER row above)
6. base rate reported next to metric?          YES - 0.188 (positive rate, grouped test set)

wrote work/outputs/w06_metrics.json


**The trap still works exactly as it should.** Adding `ctr_gap` — the column the label is
literally thresholded from — pushes AUC from 0.798 to a suspiciously perfect **1.0**. That's the
textbook symptom from Week 3: a near-perfect score is a leak alert, not a win. It's removed, and the
honest AUC to report anywhere is **0.798**.

The rest of the checklist comes back clean: no label-derived features, no future-window columns
(everything is the March 2026 window used to build the label itself), no product/flag columns (none
exist in this warehouse release), the visibility filter (`impressions >= 150`) doesn't depend on the
outcome, the split is grouped by client with zero overlap, and the base rate (18.8%) is reported
next to every metric above rather than left implicit.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

On the grouped-by-client split — the design that best approximates
scoring a client the model hasn't seen — the logistic regression's ranked queue showed **measured**
precision@50 of 0.48 and precision@100 of 0.52 against a base rate of 0.188, compared to the
Week-4 rule-based baseline's 0.16/0.20 on the same split and label. This is a **directional**
result, not a guarantee: it reflects one month (March 2026), one label definition (bottom decile of
tier-adjusted CTR gap), and 11 held-out clients. It supports treating the model's ranked queue as a
**decision-support** tool for prioritizing which pages an editor reviews first — it does not
establish that rewriting a flagged page's title/meta will recover clicks, since that would require
an actual before/after intervention, not this observational comparison.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Claim before -> after, stated as data:")
print("before: 'both models beat the baseline' (no split/window/label caveat attached)")
print("after:  measured precision@50 0.48 vs 0.16 baseline, on the grouped March-2026 split,")
print("        against an 18.8% base rate -- directional, decision-support, not causal")

Claim before -> after, stated as data:
before: 'both models beat the baseline' (no split/window/label caveat attached)
after:  measured precision@50 0.48 vs 0.16 baseline, on the grouped March-2026 split,
        against an 18.8% base rate -- directional, decision-support, not causal


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.